# Task 1

In [1]:
import numpy as np
import pandas as pd
import Logreg

### Setting up data

In [2]:
train_set = pd.read_csv("D:/Machine Learning/ML kaggle/train.csv")
train_features = pd.read_csv("D:/Machine Learning/ML kaggle/train_features.csv")
# test_features = pd.read_csv("D:/Machine Learning/ML kaggle/test_features.csv")
# test_set = pd.read_csv("D:/Machine Learning/ML kaggle/test.csv")

shape = train_features.shape
print(shape)
# print(train_set.loc[0, 'text'])
print(train_features.columns)

Y = train_features["label"].to_numpy(dtype=np.float32)
X = train_features.drop(columns=["id", "label"]).values

X_train = X[:16000]
Y_train = Y[:16000]
Y_test = Y[16000:]
X_test = X[16000:]

print(X.shape)
print(Y.shape)

(20000, 5002)
Index(['id', 'label', '1', '2', '3', '4', '5', '6', '7', '8',
       ...
       '4991', '4992', '4993', '4994', '4995', '4996', '4997', '4998', '4999',
       '5000'],
      dtype='str', length=5002)
(20000, 5000)
(20000,)


Sparsity check

In [17]:
total_elements = X_train.size

# Count how many cells contain exactly 0
zero_elements = total_elements - np.count_nonzero(X_train)

# Calculate the sparsity percentage
sparsity_level = (zero_elements / total_elements) * 100

print(f"Matrix Sparsity Level: {sparsity_level:.2f}%")
print(f"Matrix Density Level: {100 - sparsity_level:.2f}%")

Matrix Sparsity Level: 98.64%
Matrix Density Level: 1.36%


### Run Logistic Regression

Full-batch Gradient Descent training

In [3]:
X_train_scaled, train_max_abs = Logreg.maxabs_scale(X_train)
X_test_scaled, _ = Logreg.maxabs_scale(X_test, train_max_abs)

In [8]:
best_cost, best_theta, best_bias = Logreg.logreg_full_batch(X_train_scaled, Y_train, 3000, 0.1)    # epoch size, learning rate

Epoch 3000: New best cost = 0.6931
Epoch 2999: New best cost = 0.6910
Epoch 2998: New best cost = 0.6890
Epoch 2997: New best cost = 0.6871
Epoch 2996: New best cost = 0.6853
Epoch 2995: New best cost = 0.6837
Epoch 2994: New best cost = 0.6820
Epoch 2993: New best cost = 0.6805
Epoch 2992: New best cost = 0.6791
Epoch 2991: New best cost = 0.6777
Epoch 2990: New best cost = 0.6764
Epoch 2989: New best cost = 0.6752
Epoch 2988: New best cost = 0.6740
Epoch 2987: New best cost = 0.6729
Epoch 2986: New best cost = 0.6719
Epoch 2985: New best cost = 0.6709
Epoch 2984: New best cost = 0.6699
Epoch 2983: New best cost = 0.6690
Epoch 2982: New best cost = 0.6682
Epoch 2981: New best cost = 0.6673
Epoch 2980: New best cost = 0.6666
Epoch 2979: New best cost = 0.6658
Epoch 2978: New best cost = 0.6651
Epoch 2977: New best cost = 0.6644
Epoch 2976: New best cost = 0.6638
Epoch 2975: New best cost = 0.6632
Epoch 2974: New best cost = 0.6626
Epoch 2973: New best cost = 0.6620
Epoch 2972: New best

Mini-batch Gradient Descent training

In [5]:
best_cost1, best_theta1, best_bias1 = Logreg.logreg_mini_batch(X_train_scaled, Y_train, 100, 512, 0.1, 42)   # epoch size, batch size, learning rate, seed

Epoch 100: New best cost= 0.6596
Epoch 99: New best cost= 0.6507
Epoch 98: New best cost= 0.6458
Epoch 97: New best cost= 0.6418
Epoch 96: New best cost= 0.6381
Epoch 95: New best cost= 0.6346
Epoch 94: New best cost= 0.6313
Epoch 93: New best cost= 0.6282
Epoch 92: New best cost= 0.6252
Epoch 91: New best cost= 0.6224
Epoch 90: New best cost= 0.6196
Epoch 89: New best cost= 0.6170
Epoch 88: New best cost= 0.6145
Epoch 87: New best cost= 0.6122
Epoch 86: New best cost= 0.6099
Epoch 85: New best cost= 0.6077
Epoch 84: New best cost= 0.6055
Epoch 83: New best cost= 0.6035
Epoch 82: New best cost= 0.6015
Epoch 81: New best cost= 0.5996
Epoch 80: New best cost= 0.5977
Epoch 79: New best cost= 0.5958
Epoch 78: New best cost= 0.5941
Epoch 77: New best cost= 0.5923
Epoch 76: New best cost= 0.5907
Epoch 75: New best cost= 0.5890
Epoch 74: New best cost= 0.5874
Epoch 73: New best cost= 0.5859
Epoch 72: New best cost= 0.5843
Epoch 71: New best cost= 0.5828
Epoch 70: New best cost= 0.5814
Epoch 6

Testing

In [9]:
probabilities = Logreg.sigmoid(best_theta, X_test_scaled, best_bias)
predictions = (probabilities >= 0.5).astype(int)

accuracy = np.mean(predictions == Y_test)
print(f"Model Accuracy on Test Set using Full-batch: {accuracy * 100:.2f}%")

prob = Logreg.sigmoid(best_theta1, X_test_scaled, best_bias1)
pred = (prob >= 0.5).astype(int)

acc = np.mean(pred == Y_test)
print(f"Model Accuracy on Test Set using Mini-batch: {acc * 100:.2f}%")

Model Accuracy on Test Set using Full-batch: 71.53%
Model Accuracy on Test Set using Mini-batch: 71.92%


In [7]:
print("theta norm:", np.linalg.norm(best_theta))
print("theta max/min:", best_theta.max(), best_theta.min())
print("bias:", best_bias)
print("prediction distribution:", np.unique(predictions, return_counts=True))

theta norm: 0.43032824739086933
theta max/min: 0.061651950201938366 -0.04056457380562928
bias: 0.4078328450218132
prediction distribution: (array([0, 1]), array([   5, 3995]))


## Benchmark with sklearn

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_train_SK, X_val_SK, y_train_SK, y_val_SK = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

sklearn_model = LogisticRegression(max_iter=1000, random_state=42)
sklearn_model.fit(X_train_SK, y_train_SK)
sklearn_val_preds = sklearn_model.predict(X_val_SK)
sklearn_accuracy = np.mean(sklearn_val_preds == y_val_SK)
print(f"Scikit-Learn Validation Accuracy: {sklearn_accuracy * 100:.2f}%")

Scikit-Learn Validation Accuracy: 74.78%
